<h1>RAZ Systems </h1>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("raz/ajaz.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

With over 30 years of experience in the technology and research sectors, I bring a 
rich blend of academic, technical, and leadership expertise. My journey began in 
Computer Science, with early roles at the Raman Research Institute and as a visiting
lecturer at Bangalore University. I then coordinated GIS initiatives in Qatar, 
focusing on land surveys and building permits. As a Team Lead at Compaq Computers, I
specialized in Product Information Management (PIM). Currently, I manage operations 
at Raz Systems and serve as a Software Engineer at the International Organization, 
leading financial software development initiatives.


In [5]:
with open("raz/RazSystems.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Ajaz"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

"You are acting as Ajaz. You are answering questions on Ajaz's website, particularly questions related to Ajaz's career, background, skills and experience. Your responsibility is to represent Ajaz for interactions on the website as faithfully as possible. You are given a summary of Ajaz's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nWebsite Design and Development\nYou'd be surprised at how happy the right CMS can make you. Raz Systems's experience in designing, developing, and supporting Joomla, Drupal, and WordPress websites gives us the insight to recommend the best solution for you.\n\nTraining & Consulting Focus\nRaz Systems provides consulting, development, and training services across Data Science, Big Data Engineering, Machine Learning, LLM Applications, and Agentic AI. Our team helps org

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [28]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [12]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [13]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [14]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [15]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [16]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [17]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "are you a programmer?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [18]:
reply

"Yes, I am a programmer with over 30 years of experience in technology and research. My background includes software development, particularly in financial software. I also have expertise in various programming languages and technologies, which enables me to manage projects effectively and lead teams in developing robust solutions. If you're looking for specific programming services or have any questions about programming, feel free to ask!"

In [19]:
evaluate(reply, "are you a programmer?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent directly answers the question, drawing accurately from the provided context (LinkedIn profile and summary). It maintains a professional and engaging tone, reinforcing Ajaz's experience and inviting further inquiry, which aligns perfectly with the persona instructions.")

In [21]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [22]:
def chat(message, history):
    if "programmer" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [27]:


gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [25]:
import gradio
print(gradio.__version__)
print(gradio.__file__)  # shows the actual file path being loaded

6.14.0
d:\git\agentic\.venv\Lib\site-packages\gradio\__init__.py


In [26]:
import inspect
print(inspect.signature(gr.ChatInterface.__init__))

(self, fn: 'Callable', *, multimodal: 'bool' = False, chatbot: 'Chatbot | None' = None, textbox: 'Textbox | MultimodalTextbox | None' = None, additional_inputs: 'str | Component | list[str | Component] | None' = None, additional_inputs_accordion: 'str | Accordion | None' = None, additional_outputs: 'Component | list[Component] | None' = None, editable: 'bool' = False, examples: 'list[str] | list[MultimodalValue] | list[list] | None' = None, example_labels: 'list[str] | None' = None, example_icons: 'list[str] | None' = None, run_examples_on_click: 'bool' = True, cache_examples: 'bool | None' = None, cache_mode: "Literal['eager', 'lazy'] | None" = None, title: 'str | I18nData | None' = None, description: 'str | None' = None, flagging_mode: "Literal['never', 'manual'] | None" = None, flagging_options: 'list[str] | tuple[str, ...] | None' = ('Like', 'Dislike'), flagging_dir: 'str' = '.gradio/flagged', analytics_enabled: 'bool | None' = None, autofocus: 'bool' = True, autoscroll: 'bool' = T

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://www.razsystems.com/"
page = requests.get(URL)

soup = BeautifulSoup(page.content, "html.parser")
soup


<!-- header start -->
<!DOCTYPE html>

<html>
<head>
<!-- Required meta tags -->
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
<title>Raz Systems, Software Company</title>
<!-- Bootstrap CSS -->
<link href="css/bootstrap.min.css" rel="stylesheet"/>
<link href="https://fonts.googleapis.com/css2?family=Montserrat:wght@100;200;300;400;500;600;700;800;900&amp;display=swap" rel="stylesheet"/>
<link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.css" rel="stylesheet"/>
<link href="css/owl.carousel.min.css" rel="stylesheet"/>
<link href="css/aos.css" rel="stylesheet"/>
<link href="css/main.css" rel="stylesheet"/>
<link href="css/style.css" rel="stylesheet"/>
<meta charset="utf-8">
<title>Review Our Courses</title>
<meta content="width=device-width, initial-scale=1" name="viewport">
<link crossorigin="anonymous" href="https://maxcdn.bootstrapcdn.com/bootstrap/4.0.0/css/bootstrap.min.css" integr